# NB-B - Watcher du dataset

Le watcher observe uniquement les versions publiees et taguees `auto-feedback`. Pour chaque nouvelle version, il clone le pipeline et lui passe l ID exact du dataset.


In [1]:
!pip -q install clearml python-dotenv



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# La configuration ClearML vient de variables d'environnement ou du fichier .env local.
# Ne mettez jamais de cles API dans le notebook.
import os
from pathlib import Path

try:
    from dotenv import load_dotenv, find_dotenv
    env_file = find_dotenv(usecwd=True)
    if env_file:
        load_dotenv(env_file)
    else:
        load_dotenv()
except ImportError:
    pass

REQUIRED_CLEARML_VARS = (
    "CLEARML_WEB_HOST",
    "CLEARML_API_HOST",
    "CLEARML_FILES_HOST",
    "CLEARML_API_ACCESS_KEY",
    "CLEARML_API_SECRET_KEY",
)
missing = [name for name in REQUIRED_CLEARML_VARS if not os.getenv(name)]
if missing:
    raise RuntimeError(
        "Configuration ClearML absente. Renseignez .env ou les variables d'environnement : "
        + ", ".join(missing)
    )
print("Configuration ClearML chargee depuis l'environnement.")


Configuration ClearML chargee depuis l'environnement.


In [ ]:
# NB-B - Watcher ClearML : un dataset publie declenche un run reproductible
import json
import os
import time
import traceback
from pathlib import Path

from clearml import Dataset, Task

PROJECT = "Image-Classifier/Gradio-Retrain-Demo-IABD"
DATASET_NAME = "image-feedback"
PIPELINE_TASK_ID = os.getenv("PIPELINE_TASK_ID", "").strip()
if not PIPELINE_TASK_ID:
    candidates = Task.get_tasks(
        task_name="retrain_pipeline",
        task_filter={"status": ["completed", "published"]},
    ) or []
    if candidates:
        PIPELINE_TASK_ID = candidates[-1].id

if not PIPELINE_TASK_ID:
    raise RuntimeError("Definissez PIPELINE_TASK_ID avec l'ID de la premiere tache retrain_pipeline.")

QUEUE = "controller"
POLL_SECS = 15
WORK_DIR = Path(os.getenv("MLOPS_WORK_DIR", Path.cwd() / "mlops_feedback"))
STATE_FILE = WORK_DIR / "watcher_state.json"

WORK_DIR.mkdir(parents=True, exist_ok=True)
controller_task = Task.init(
    project_name=PROJECT,
    task_name="dataset_watcher",
    task_type=Task.TaskTypes.controller,
    reuse_last_task_id=False,
)
print(f"Watcher {controller_task.id} : datasets '{DATASET_NAME}' -> pipeline {PIPELINE_TASK_ID}")

def _read_state() -> dict:
    try:
        return json.loads(STATE_FILE.read_text(encoding="utf-8"))
    except Exception:
        return {"initialized": False, "last_dataset_id": ""}

def _write_state(initialized: bool, dataset_id: str) -> None:
    STATE_FILE.write_text(
        json.dumps({"initialized": initialized, "last_dataset_id": dataset_id}),
        encoding="utf-8",
    )

def _latest_feedback_dataset():
    datasets = Dataset.list_datasets(
        dataset_project=PROJECT,
        partial_name=DATASET_NAME,
        tags=["auto-feedback"],
        only_completed=True,
        recursive_project_search=False,
    ) or []
    matching = [item for item in datasets if item.get("name") == DATASET_NAME]
    if not matching:
        return None
    return max(matching, key=lambda item: item.get("created", ""))

def _enqueue_pipeline(dataset_id: str) -> str:
    source = Task.get_task(task_id=PIPELINE_TASK_ID)
    run = Task.clone(
        source_task=source,
        name=f"{source.name}_dataset_{dataset_id[:8]}_{int(time.time())}",
    )
    # Le parametre est lu par le decorateur Pipeline dans le notebook C.
    # Il lie ce run au dataset qui l'a declenche, sans course vers "le plus recent".
    run.set_parameter("Datasets/feedback_dataset_id", dataset_id)
    Task.enqueue(task=run, queue_name=QUEUE)
    return run.id

state = _read_state()
initialized = bool(state.get("initialized", False))
last_seen = str(state.get("last_dataset_id", ""))
print("Dernier dataset traite :", last_seen or "aucun")
print(f"Surveillance toutes les {POLL_SECS} secondes. Ctrl+C pour arreter.")

try:
    while True:
        latest = _latest_feedback_dataset()
        if latest is None:
            print("[watcher] Aucun dataset de feedback publie.")
            if not initialized:
                # Si le watcher demarre avant le premier feedback, le prochain dataset doit declencher.
                initialized = True
                _write_state(initialized, last_seen)
        else:
            dataset_id = latest["id"]
            if not initialized:
                # On initialise le curseur : les anciennes versions ne doivent pas relancer le systeme.
                last_seen = dataset_id
                initialized = True
                _write_state(initialized, dataset_id)
                print(f"[watcher] Curseur initialise sur {dataset_id}.")
            elif dataset_id != last_seen:
                run_id = _enqueue_pipeline(dataset_id)
                last_seen = dataset_id
                _write_state(initialized, dataset_id)
                print(f"[watcher] Pipeline {run_id} enfile pour dataset {dataset_id}.")
            else:
                print(f"[watcher] Rien de nouveau : {dataset_id}.")
        time.sleep(POLL_SECS)
except KeyboardInterrupt:
    print("Watcher arrete par l'utilisateur.")
except Exception:
    print("Erreur watcher :\n", traceback.format_exc())
    raise
finally:
    controller_task.close()


ClearML Task: created new task id=0df3469865d540379a76f1fef270ac79
ClearML results page: https://app.clear.ml/projects/e3ab23ee4564441799ef004832299385/tasks/0df3469865d540379a76f1fef270ac79/output/log
Watcher 0df3469865d540379a76f1fef270ac79 : datasets 'image-feedback' -> pipeline 81872406a4244720a00a35dffe8a7b28
Dernier dataset traite : aucun
Surveillance toutes les 15 secondes. Ctrl+C pour arreter.
[watcher] Aucun dataset de feedback publie.
[watcher] Aucun dataset de feedback publie.
[watcher] Aucun dataset de feedback publie.
[watcher] Aucun dataset de feedback publie.
[watcher] Aucun dataset de feedback publie.
[watcher] Aucun dataset de feedback publie.
[watcher] Aucun dataset de feedback publie.
[watcher] Aucun dataset de feedback publie.
[watcher] Aucun dataset de feedback publie.
[watcher] Aucun dataset de feedback publie.
[watcher] Aucun dataset de feedback publie.
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start
[watche